# CADGenesis-Mini — a real, trainable seed of CADGenesis-LM

This notebook is a **working, scaled-down prototype**, not the full mega-architecture from
the earlier design doc. It's built so you can actually run and train it end-to-end on a
free Colab GPU (or even CPU) in a few minutes.

**What it honestly includes:**
- A hybrid tokenizer: a language token stream + a geometry/feature token stream (with
  quantized numeric parameters), matching the "hybrid tokenizer" idea from Phase 1.
- A small encoder–decoder Transformer where the decoder (geometry) cross-attends to the
  encoder (language) — a real, working instance of "geometry attention," plus a token-type
  embedding standing in for the "token hierarchy" concept.
- A synthetic paired dataset (text → CAD construction sequence), since no real CAD corpus
  is attached here.
- A training loop, greedy decoding, a structural/geometric validator, and a matplotlib
  renderer — a small, real version of the "CAD execution flow" (generate → validate → render).

**What it deliberately does *not* attempt** (these need real infrastructure, API keys/cost,
or multi-node clusters, and would not actually run in Colab even if we wrote the code):
the 8-agent orchestration system, 7-teacher distillation against commercial LLM APIs, the
graph/vector memory subsystems, EWC-based continual learning, and distributed/Kubernetes
training and deployment. Those stay as architecture (Phase 1 doc) until you have the
infra to back them.

**Runtime:** Runtime → Change runtime type → GPU (T4 is plenty). CPU also works, just slower.


In [ ]:
!pip -q install torch --upgrade

## Phase A — Hybrid Tokenizer + Synthetic CAD Dataset

- `NUM_BINS` quantizes continuous parameters into discrete bins → `NUM_k` tokens
  (this is how real CAD-sequence models like DeepCAD/SkexGen represent geometry too).
- Four toy primitive/feature types: `BOX`, `CYLINDER`, `SPHERE`, and a `SKETCH_RECT` +
  `EXTRUDE` feature pair (sketch-then-extrude, like a real feature tree).
- `LangTokenizer` is a minimal word-level tokenizer built from the synthetic text corpus.


In [ ]:
"""
CADGenesis-Mini: hybrid tokenizer + synthetic CAD dataset.

This is a deliberately small, honest, working slice of the "hybrid tokenizer"
concept from the Phase 1 architecture doc: a LANGUAGE token stream and a
GEOMETRY/FEATURE token stream, generated from paired (text, CAD-sequence)
data.

CAD objects are represented the way real CAD-generation research does it
(see DeepCAD / SkexGen-style sequence representations): a short sequence of
primitive/feature tokens interleaved with *quantized* numeric parameter
tokens, rather than continuous CAD files. This makes the target a clean,
learnable token stream instead of pixels or raw B-Rep.
"""
import random
import re

# ---------------------------------------------------------------------------
# Numeric quantization (mirrors the "geometry token generation" idea from
# Phase 1: continuous parameters -> discrete bins -> tokens)
# ---------------------------------------------------------------------------
NUM_BINS = [round(0.5 + 0.5 * i, 2) for i in range(20)]  # 0.5 .. 10.0 step 0.5


def value_to_bin(v):
    return min(range(len(NUM_BINS)), key=lambda i: abs(NUM_BINS[i] - v))


def sample_value():
    return random.choice(NUM_BINS)


# ---------------------------------------------------------------------------
# CAD token vocabulary (the "geometry/feature token" side of the tokenizer)
# ---------------------------------------------------------------------------
CAD_SPECIALS = ["<pad>", "<bos>", "<eos>"]
CAD_PRIMITIVES = ["BOX", "CYLINDER", "SPHERE", "SKETCH_RECT", "EXTRUDE"]
CAD_NUMS = [f"NUM_{i}" for i in range(len(NUM_BINS))]

CAD_VOCAB = CAD_SPECIALS + CAD_PRIMITIVES + CAD_NUMS
CAD_TOK2ID = {t: i for i, t in enumerate(CAD_VOCAB)}
CAD_ID2TOK = {i: t for t, i in CAD_TOK2ID.items()}

PAD_ID, BOS_ID, EOS_ID = CAD_TOK2ID["<pad>"], CAD_TOK2ID["<bos>"], CAD_TOK2ID["<eos>"]


# Token "type" ids used for the hierarchy/type embedding in the model:
# 0 = special, 1 = primitive/feature token, 2 = numeric parameter token
def cad_token_type(tok_id):
    tok = CAD_ID2TOK[tok_id]
    if tok in CAD_SPECIALS:
        return 0
    if tok in CAD_PRIMITIVES:
        return 1
    return 2


# ---------------------------------------------------------------------------
# Shape generators: each returns (text, cad_token_list[no bos/eos])
# ---------------------------------------------------------------------------
BOX_TEMPLATES = [
    "Create a box that is {w} units wide, {h} units tall, and {d} units deep.",
    "Design a rectangular block with width {w}, height {h}, and depth {d}.",
    "I need a box sized {w} by {h} by {d}.",
    "Make a solid box, width {w}, height {h}, depth {d}.",
]
CYLINDER_TEMPLATES = [
    "Create a cylinder with radius {r} and height {h}.",
    "Make a cylindrical part {h} units tall with a radius of {r}.",
    "Design a cylinder, radius {r}, height {h}.",
]
SPHERE_TEMPLATES = [
    "Create a sphere with radius {r}.",
    "Design a ball of radius {r}.",
    "Make a spherical part with radius {r}.",
]
EXTRUDE_TEMPLATES = [
    "Sketch a rectangle {w} by {h} and extrude it {depth} units.",
    "Create an extruded block from a {w} by {h} rectangle, extruded {depth} units deep.",
    "Draw a {w} by {h} rectangle and extrude {depth} units.",
]


def gen_box():
    w, h, d = sample_value(), sample_value(), sample_value()
    text = random.choice(BOX_TEMPLATES).format(w=w, h=h, d=d)
    seq = ["BOX", f"NUM_{value_to_bin(w)}", f"NUM_{value_to_bin(h)}", f"NUM_{value_to_bin(d)}"]
    return text, seq


def gen_cylinder():
    r, h = sample_value(), sample_value()
    text = random.choice(CYLINDER_TEMPLATES).format(r=r, h=h)
    seq = ["CYLINDER", f"NUM_{value_to_bin(r)}", f"NUM_{value_to_bin(h)}"]
    return text, seq


def gen_sphere():
    r = sample_value()
    text = random.choice(SPHERE_TEMPLATES).format(r=r)
    seq = ["SPHERE", f"NUM_{value_to_bin(r)}"]
    return text, seq


def gen_extrude():
    w, h, depth = sample_value(), sample_value(), sample_value()
    text = random.choice(EXTRUDE_TEMPLATES).format(w=w, h=h, depth=depth)
    seq = [
        "SKETCH_RECT",
        f"NUM_{value_to_bin(w)}",
        f"NUM_{value_to_bin(h)}",
        "EXTRUDE",
        f"NUM_{value_to_bin(depth)}",
    ]
    return text, seq


GENERATORS = [gen_box, gen_cylinder, gen_sphere, gen_extrude]


def generate_example():
    text, seq = random.choice(GENERATORS)()
    cad_ids = [BOS_ID] + [CAD_TOK2ID[t] for t in seq] + [EOS_ID]
    return text, cad_ids


# ---------------------------------------------------------------------------
# Language tokenizer: small word-level vocab built from the synthetic corpus
# ---------------------------------------------------------------------------
_word_re = re.compile(r"[a-zA-Z]+|\d+\.\d+|\d+|[.,]")


def word_tokenize(text):
    return _word_re.findall(text.lower())


class LangTokenizer:
    def __init__(self):
        self.tok2id = {"<pad>": 0, "<unk>": 1}
        self.id2tok = {0: "<pad>", 1: "<unk>"}

    def build_vocab(self, texts):
        vocab = set()
        for t in texts:
            vocab.update(word_tokenize(t))
        for w in sorted(vocab):
            if w not in self.tok2id:
                idx = len(self.tok2id)
                self.tok2id[w] = idx
                self.id2tok[idx] = w

    def encode(self, text):
        return [self.tok2id.get(w, 1) for w in word_tokenize(text)]

    def __len__(self):
        return len(self.tok2id)


def build_dataset(n, lang_tok=None):
    """Generate n (text, cad_ids) pairs. If lang_tok has no vocab yet, build it."""
    raw = [generate_example() for _ in range(n)]
    texts = [t for t, _ in raw]
    if lang_tok is not None and len(lang_tok) <= 2:
        lang_tok.build_vocab(texts)
    return raw

In [ ]:
# quick sanity check
lang_tok = LangTokenizer()
sample = build_dataset(5, lang_tok=lang_tok)
for text, cad_ids in sample:
    print(text)
    print(" ->", [CAD_ID2TOK[i] for i in cad_ids])
print("lang vocab size:", len(lang_tok))
print("cad vocab size:", len(CAD_VOCAB))

## Phase B — Geometry-Aware Transformer (scaled down)

Encoder = language tokens. Decoder = CAD tokens, cross-attending to the encoder output,
plus a type embedding (special / primitive / parameter) added to each CAD token embedding.


In [ ]:
"""
CADGenesis-Mini model.

A small, real, trainable encoder-decoder Transformer:
  - Encoder reads LANGUAGE tokens (the design request).
  - Decoder generates GEOMETRY/FEATURE tokens (the CAD construction sequence),
    cross-attending to the language encoding -- this is the working, scaled
    version of "Geometry Attention" from the Phase 1 architecture: geometry
    generation is literally conditioned on language context via attention.
  - CAD tokens get an extra TYPE embedding (special / primitive / parameter),
    a small real instance of the "token hierarchy" idea: the model gets an
    explicit signal about what *kind* of token it's producing next, not just
    which token.

This is intentionally small enough to train on a single Colab GPU (or even
CPU) in minutes, not a claim to be the full CADGenesis-LM architecture.
"""
import math

import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class CADGenesisMini(nn.Module):
    def __init__(
        self,
        lang_vocab_size,
        cad_vocab_size,
        d_model=128,
        nhead=4,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=256,
        dropout=0.1,
        max_len=64,
    ):
        super().__init__()
        self.d_model = d_model

        # --- Language ("text") token embedding ---
        self.lang_embed = nn.Embedding(lang_vocab_size, d_model, padding_idx=0)

        # --- CAD/geometry token embedding + type (hierarchy) embedding ---
        self.cad_embed = nn.Embedding(cad_vocab_size, d_model, padding_idx=0)
        self.type_embed = nn.Embedding(3, d_model)  # 0=special 1=primitive 2=parameter

        self.pos_enc = PositionalEncoding(d_model, max_len)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.out_proj = nn.Linear(d_model, cad_vocab_size)

    def forward(
        self,
        src_ids,
        tgt_in_ids,
        tgt_type_ids,
        src_key_padding_mask=None,
        tgt_key_padding_mask=None,
    ):
        """
        src_ids:        (B, S)  language token ids
        tgt_in_ids:      (B, T)  CAD token ids, decoder input (shifted right)
        tgt_type_ids:    (B, T)  CAD token *type* ids for the same positions
        """
        src = self.pos_enc(self.lang_embed(src_ids) * math.sqrt(self.d_model))
        tgt = self.cad_embed(tgt_in_ids) + self.type_embed(tgt_type_ids)
        tgt = self.pos_enc(tgt * math.sqrt(self.d_model))

        T = tgt_in_ids.size(1)
        # bool causal mask (kept the same dtype as the padding masks to avoid
        # PyTorch's mismatched-mask-type warning/slow path)
        causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt.device), diagonal=1)

        out = self.transformer(
            src,
            tgt,
            tgt_mask=causal_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.out_proj(out)  # (B, T, cad_vocab_size)

In [ ]:
# quick shape check
_lt = LangTokenizer()
_ = build_dataset(50, lang_tok=_lt)
m = CADGenesisMini(lang_vocab_size=len(_lt), cad_vocab_size=len(CAD_VOCAB))
print(m)
n_params = sum(p.numel() for p in m.parameters())
print(f"parameters: {n_params:,}")

## Phase C — Training Flow

Teacher-forced next-token prediction on the CAD sequence, conditioned on the language
encoding. Loss is cross-entropy ignoring `<pad>`. Increase `n_train` / `epochs` for
better numeric accuracy — this default config trains in under a minute on a T4.


In [ ]:
import torch.nn as nn
from model import CADGenesisMini
from torch.utils.data import DataLoader, Dataset

from data import (
    BOS_ID,
    CAD_TOK2ID,
    CAD_VOCAB,
    EOS_ID,
    PAD_ID,
    LangTokenizer,
    build_dataset,
    cad_token_type,
)

random.seed(0)
torch.manual_seed(0)


class CADDataset(Dataset):
    def __init__(self, pairs, lang_tok, max_src=32, max_tgt=16):
        self.pairs = pairs
        self.lang_tok = lang_tok
        self.max_src = max_src
        self.max_tgt = max_tgt

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        text, cad_ids = self.pairs[idx]
        src = self.lang_tok.encode(text)[: self.max_src]
        tgt = cad_ids[: self.max_tgt]
        return src, tgt


def collate(batch, max_src=32, max_tgt=16):
    srcs, tgts = zip(*batch, strict=False)
    B = len(batch)
    src_pad = torch.zeros(B, max_src, dtype=torch.long)
    tgt_pad = torch.full((B, max_tgt), PAD_ID, dtype=torch.long)
    for i, s in enumerate(srcs):
        src_pad[i, : len(s)] = torch.tensor(s, dtype=torch.long)
    for i, t in enumerate(tgts):
        tgt_pad[i, : len(t)] = torch.tensor(t, dtype=torch.long)
    return src_pad, tgt_pad


def run(n_train=4000, n_val=400, epochs=8, batch_size=64, device=None, verbose=True):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    lang_tok = LangTokenizer()
    train_pairs = build_dataset(n_train, lang_tok=lang_tok)
    val_pairs = build_dataset(n_val, lang_tok=lang_tok)

    train_ds = CADDataset(train_pairs, lang_tok)
    val_ds = CADDataset(val_pairs, lang_tok)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate)

    model = CADGenesisMini(lang_vocab_size=len(lang_tok), cad_vocab_size=len(CAD_VOCAB)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    type_lookup = torch.tensor([cad_token_type(i) for i in range(len(CAD_VOCAB))])

    def step(src, tgt, train=True):
        src, tgt = src.to(device), tgt.to(device)
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        tgt_type = type_lookup[tgt_in.cpu()].to(device)

        src_pad_mask = src == 0
        tgt_pad_mask = tgt_in == PAD_ID

        logits = model(
            src,
            tgt_in,
            tgt_type,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
        )
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()
        return loss.item()

    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for src, tgt in train_dl:
            total += step(src, tgt, train=True)
        train_loss = total / len(train_dl)

        model.eval()
        with torch.no_grad():
            vtotal = sum(step(src, tgt, train=False) for src, tgt in val_dl)
        val_loss = vtotal / len(val_dl)

        if verbose:
            print(f"epoch {epoch:2d}  train_loss {train_loss:.4f}  val_loss {val_loss:.4f}")

    return model, lang_tok, device

In [ ]:
model, lang_tok, device = run(n_train=8000, n_val=800, epochs=20, batch_size=128)
print("device:", device)

## Phase D — Inference + CAD Execution (generate → validate → render)

`generate()` does greedy autoregressive decoding from a text prompt.
`parse_cad_sequence()` is a real structural validator — malformed sequences are rejected,
not silently accepted (the scaled-down stand-in for the Geometry Validation gate in the
Phase 1 CAD execution flow). `render()` draws the validated primitive.


In [ ]:
"""
Inference + a minimal, real "CAD execution" step.

generate(): greedy autoregressive decoding from a text prompt -> CAD tokens.
parse_cad_sequence(): turns the token sequence back into structured geometry
    (this is the scaled-down stand-in for the Geometry Kernel / Topology
    Analysis / Geometry Validation stages from the Phase 1 CAD execution
    flow -- it's a real parser+validator, just against a toy primitive set
    instead of OpenCascade/FreeCAD).
render(): draws the resulting primitive with matplotlib, so you can visually
    confirm the model produced something geometrically sensible.
"""
import torch

from data import CAD_ID2TOK, CAD_TOK2ID, NUM_BINS


@torch.no_grad()
def generate(model, lang_tok, text, device, max_len=16):
    model.eval()
    src_ids = lang_tok.encode(text)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_pad_mask = src == 0

    tgt = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
    for _ in range(max_len - 1):
        tgt_type = torch.tensor([[cad_token_type(t.item()) for t in tgt[0]]], device=device)
        logits = model(src, tgt, tgt_type, src_key_padding_mask=src_pad_mask)
        next_id = logits[0, -1].argmax(-1).item()
        tgt = torch.cat([tgt, torch.tensor([[next_id]], device=device)], dim=1)
        if next_id == EOS_ID:
            break
    return [CAD_ID2TOK[i.item()] for i in tgt[0]]


def parse_cad_sequence(tokens):
    """
    Validate + interpret a generated token sequence. Returns
    (is_valid, shape_dict_or_error_message) -- this is the real, working
    equivalent of the "Geometry Validation" gate in the Phase 1 CAD
    execution flow: structurally invalid sequences are rejected here, not
    silently accepted.
    """
    toks = [t for t in tokens if t not in ("<bos>", "<eos>", "<pad>")]

    def num(tok):
        if not tok.startswith("NUM_"):
            return None
        idx = int(tok.split("_")[1])
        if idx >= len(NUM_BINS):
            return None
        return NUM_BINS[idx]

    if not toks:
        return False, "empty sequence"

    head = toks[0]
    if head == "BOX" and len(toks) >= 4:
        w, h, d = num(toks[1]), num(toks[2]), num(toks[3])
        if None in (w, h, d):
            return False, "malformed BOX parameters"
        return True, {"type": "box", "w": w, "h": h, "d": d}

    if head == "CYLINDER" and len(toks) >= 3:
        r, h = num(toks[1]), num(toks[2])
        if None in (r, h):
            return False, "malformed CYLINDER parameters"
        return True, {"type": "cylinder", "r": r, "h": h}

    if head == "SPHERE" and len(toks) >= 2:
        r = num(toks[1])
        if r is None:
            return False, "malformed SPHERE parameters"
        return True, {"type": "sphere", "r": r}

    if head == "SKETCH_RECT" and len(toks) >= 5 and toks[3] == "EXTRUDE":
        w, h, depth = num(toks[1]), num(toks[2]), num(toks[4])
        if None in (w, h, depth):
            return False, "malformed EXTRUDE parameters"
        return True, {"type": "extrude", "w": w, "h": h, "depth": depth}

    return False, f"unrecognized/malformed sequence starting with {head}"


def render(shape, ax=None):
    """Draw the validated shape with matplotlib (box/cylinder/sphere/extrude)."""
    import matplotlib.pyplot as plt
    import numpy as np

    if ax is None:
        fig = plt.figure(figsize=(4, 4))
        ax = fig.add_subplot(projection="3d")

    t = shape["type"]
    if t in ("box", "extrude"):
        w = shape.get("w", 1)
        h = shape.get("h", 1)
        d = shape.get("d", shape.get("depth", 1))
        x, y, z = np.indices((2, 2, 2)).astype(float)
        x, y, z = x * w, y * h, z * d
        ax.bar3d(0, 0, 0, w, h, d, alpha=0.6, shade=True)
    elif t == "cylinder":
        r, h = shape["r"], shape["h"]
        theta = np.linspace(0, 2 * np.pi, 30)
        z = np.linspace(0, h, 2)
        theta, z = np.meshgrid(theta, z)
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        ax.plot_surface(x, y, z, alpha=0.6)
    elif t == "sphere":
        r = shape["r"]
        u, v = np.mgrid[0 : 2 * np.pi : 30j, 0 : np.pi : 15j]
        x = r * np.cos(u) * np.sin(v)
        y = r * np.sin(u) * np.sin(v)
        z = r * np.cos(v)
        ax.plot_surface(x, y, z, alpha=0.6)

    ax.set_title(str(shape))
    return ax

In [ ]:
prompts = [
    "Create a box that is 3.0 units wide, 5.5 units tall, and 2.0 units deep.",
    "Make a spherical part with radius 4.5.",
    "Create a cylinder with radius 1.5 and height 6.0.",
    "Sketch a rectangle 2.0 by 3.0 and extrude it 4.0 units.",
]

import matplotlib.pyplot as plt

fig = plt.figure(figsize=(14, 4))
for i, p in enumerate(prompts):
    toks = generate(model, lang_tok, p, device)
    valid, shape = parse_cad_sequence(toks)
    print(p)
    print("  tokens:", toks)
    print("  valid:", valid, "->", shape)
    if valid:
        ax = fig.add_subplot(1, len(prompts), i + 1, projection="3d")
        render(shape, ax=ax)
plt.tight_layout()
plt.show()

## Phase E — Try your own prompt

Stick to the vocabulary the model was trained on: box / cylinder / sphere / extruded
rectangle, with numbers between 0.5 and 10.0 in steps of 0.5. (Free-form CAD language
understanding is exactly the kind of thing the full-size CADGenesis-LM would need a much
larger model + real CAD corpus for — this toy model only knows what it was trained on.)


In [ ]:
your_prompt = "Create a box that is 7.0 units wide, 1.5 units tall, and 4.0 units deep."  # @param {type:"string"}
toks = generate(model, lang_tok, your_prompt, device)
valid, shape = parse_cad_sequence(toks)
print("tokens:", toks)
print("valid:", valid, "->", shape)
if valid:
    render(shape)
    plt.show()

## Save your trained model

Downloads a checkpoint + the tokenizer vocab so you can reload it later without retraining.


In [ ]:
import json

import torch

torch.save(model.state_dict(), "cadgenesis_mini.pt")
with open("lang_vocab.json", "w") as f:
    json.dump(lang_tok.tok2id, f)

from google.colab import files

files.download("cadgenesis_mini.pt")
files.download("lang_vocab.json")

## Where this actually goes next

Real next steps that build on this working seed, roughly in order of value-per-effort:

1. **Bigger + more varied synthetic grammar** — add fillets, holes, patterns, multi-feature
   assemblies. Still no real CAD data needed, just a richer generator.
2. **Swap in a real geometry kernel** — call FreeCAD/OpenCascade (via their Python
   bindings) inside `parse_cad_sequence`/`render` instead of the toy matplotlib version,
   so validation is against real B-Rep geometry, not just structural well-formedness.
3. **Real CAD corpus** — fine-tune on an actual dataset (e.g. DeepCAD's dataset) once the
   pipeline above works, instead of synthetic templates.
4. **One retrieval memory** — a simple vector store of past user CAD requests/results,
   before attempting the full 8-tier memory system from Phase 1.
5. Only after 1–4 are solid: revisit multi-agent orchestration and multi-teacher
   distillation, which need real infrastructure and budget to be worth building.
